# trainedml - Comparaison de modèles et rapport EDA

Ce notebook montre les deux fonctionnalités phares de **trainedml** :

1. `compare()` - comparer tous les modèles adaptés à un dataset en **une ligne**, par validation croisée ;
2. `Visualizer.report()` - générer un **rapport exploratoire HTML auto-contenu**.

## 1. Comparer tous les modèles en une ligne

Le type de tâche est détecté automatiquement, le prétraitement est réentraîné à chaque pli (aucune fuite d'information), et le résultat est un DataFrame trié du meilleur au moins bon.

In [ ]:
from trainedml import compare

df = compare(dataset="wine", cv=5, show_progress=False)
df.round(3)

## Comparer vos propres modèles (y compris scikit-learn)

In [ ]:
from sklearn.svm import SVC
from trainedml.models import KNNModel, RandomForestModel

df = compare(
    dataset="iris",
    models={
        "svc_rbf": SVC(kernel="rbf"),
        "knn_3": KNNModel(n_neighbors=3),
        "rf_100": RandomForestModel(n_estimators=100),
    },
    cv=5,
    show_progress=False,
)
df.round(3)

## Régression : mêmes outils, métriques adaptées

Avec une cible continue, `compare()` sélectionne automatiquement les régresseurs et retourne r², MSE, RMSE, MAE.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 200
X = pd.DataFrame({"surface": rng.uniform(20, 150, n), "pieces": rng.integers(1, 6, n)})
y = pd.Series(2000 * X["surface"] + 5000 * X["pieces"] + rng.normal(0, 8000, n), name="prix")

compare(X=X, y=y, cv=5, show_progress=False).round(3)

## 2. Rapport EDA HTML en une ligne

Le rapport contient : aperçu, statistiques descriptives, valeurs manquantes, corrélations avec heatmap, distributions, outliers (IQR), tests de normalité (Shapiro-Wilk) et VIF. Il est auto-contenu : les figures sont embarquées en base64.

In [ ]:
from trainedml.data.loader import DataLoader
from trainedml.visualization import Visualizer

X_iris, y_iris = DataLoader().load_dataset(name="iris")
data = pd.concat([X_iris, y_iris], axis=1)

viz = Visualizer(data)
html = viz.report("rapport_iris.html", title="Rapport EDA - Iris")
print("Rapport écrit dans rapport_iris.html")

Les visualisations restent aussi accessibles individuellement :

In [ ]:
fig = viz.heatmap()  # matrice de corrélation
fig